# S05b: Model-choice override record (model_registry.json)

Appends a human-approved `model_choice_override` section, read from `config/override.yaml`, to the registry under `config.databricks.artifacts_root`. **Selection is not re-run.** The locked content and `oot_evaluation` are not modified; this is asserted with hashes taken before and after the write. The override is only checked against the rule results E13 already recorded.

* **Refuses** if the section already exists with different content. A change needs a new approved entry.
* **Idempotent**: identical content is not rewritten.
* **Outputs**: `model_registry.json` key `model_choice_override`, a pre-write backup in `registry_history/`, `reports/run_summaries/s05b_model_choice_override.json`, and one MLflow run.
* **Downstream modules** (E16 onward) read the choice with `common.resolve_selected_model(registry)`.

In [0]:
# Same pinned set as s05_oot / s06b (the project has no requirements.lock yet)
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" openpyxl --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, time, hashlib, shutil, tempfile, importlib
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import yaml
import openpyxl

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)                     # no randomness used; set by convention
CONFIG_HASH = common.config_hash(cfg)
GIT_COMMIT = common.git_commit(PROJECT_ROOT)
RUN_CTX = common.print_run_context(cfg, PROJECT_ROOT)

ARTIFACTS     = common.artifacts_path(cfg)
REGISTRY_PATH = os.path.join(ARTIFACTS, "model_registry.json")
HISTORY_DIR   = os.path.join(ARTIFACTS, "registry_history")
RUN_SUMS      = os.path.join(ARTIFACTS, "reports", "run_summaries")
SUMMARY_PATH  = os.path.join(RUN_SUMS, "s05b_model_choice_override.json")
OVERRIDE_PATH = os.path.join(PROJECT_ROOT, "config", "override.yaml")
SECTION, FIELDS = common.OVERRIDE_SECTION, common.OVERRIDE_FIELDS
MODELS = ["A", "B", "C"]

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_obj(o):
    return hashlib.sha256(json.dumps(o, sort_keys=True, default=str).encode()).hexdigest()

def write_json(obj, dst):
    """Local temp file then copy to the UC volume (FUSE does not allow chmod/utime)."""
    tmp = os.path.join(tempfile.mkdtemp(prefix="s05b_"), os.path.basename(dst))
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2, default=str)
    shutil.copyfile(tmp, dst)

_t0 = time.time()
RUN_TIME = datetime.now().isoformat(timespec="seconds")
print(f"Registry    : {REGISTRY_PATH}")
print(f"Override    : {OVERRIDE_PATH}")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]}")

In [0]:
# Stop if the file is missing, has unknown/missing fields, or any field is empty
assert os.path.exists(OVERRIDE_PATH), f"STOP: {OVERRIDE_PATH} not found"
with open(OVERRIDE_PATH) as fh:
    OV_RAW = yaml.safe_load(fh) or {}
OVERRIDE_FILE_SHA = sha256_file(OVERRIDE_PATH)

def _empty(v):
    return v is None or (isinstance(v, str) and not v.strip()) or (isinstance(v, (list, dict)) and len(v) == 0) \
        or (isinstance(v, list) and any(_empty(x) for x in v))

missing = [f for f in FIELDS if f not in OV_RAW]
empty   = [f for f in FIELDS if f in OV_RAW and _empty(OV_RAW[f])]
extra   = [f for f in OV_RAW if f not in FIELDS]
assert not missing, f"STOP: override.yaml is missing field(s): {missing}"
assert not empty,   f"STOP: override.yaml has empty field(s): {empty}"
assert not extra,   f"STOP: override.yaml has unknown field(s): {extra} (expected exactly {list(FIELDS)})"

# Normalise types (dates may be parsed by YAML; lists must be lists)
OV = {f: OV_RAW[f] for f in FIELDS}
OV["approved_date"] = str(OV["approved_date"])
for f in ("rules_failed", "evidence_refs"):
    assert isinstance(OV[f], list), f"STOP: override.yaml {f} must be a list"
OV["selected_model"] = str(OV["selected_model"]).strip().upper()
OV["overrides_rule_selection_of"] = str(OV["overrides_rule_selection_of"]).strip().upper()
OV["rules_failed"] = [str(r).strip().upper() for r in OV["rules_failed"]]
datetime.strptime(OV["approved_date"], "%Y-%m-%d")       # ISO date or ValueError

for f in FIELDS:
    v = OV[f]
    print(f"  {f:<28}: {(str(v)[:110] + '...') if len(str(v)) > 110 else v}")
print(f"override.yaml sha256: {OVERRIDE_FILE_SHA}")

In [0]:
with open(REGISTRY_PATH) as fh:
    REG0 = json.load(fh)
assert REG0.get("locked") is True, "STOP: model_registry.json is not locked (E12)"
EV0 = REG0.get("oot_evaluation")
assert EV0 and EV0.get("selected_model") in MODELS, "STOP: registry has no oot_evaluation.selected_model (run E13 first)"

FILE_SHA_BEFORE   = sha256_file(REGISTRY_PATH)
LOCKED_SHA_BEFORE = common.registry_locked_sha256(REG0)
OOTEV_SHA_BEFORE  = sha256_obj(EV0)
assert LOCKED_SHA_BEFORE == EV0["locked_content_sha256"], (
    f"STOP: locked content changed since E13 ({LOCKED_SHA_BEFORE} != oot_evaluation.locked_content_sha256 "
    f"{EV0['locked_content_sha256']})")
bad = [k for k, v in REG0["files"].items() if sha256_file(v["path"]) != v["sha256"]]
assert not bad, f"STOP: {len(bad)} locked file(s) changed since the lock: {bad}"

print(f"Registry v{REG0['version']} locked at {REG0['locked_at']} | {len(REG0['files'])} locked files re-verified")
print(f"Locked content sha256 : {LOCKED_SHA_BEFORE} (= oot_evaluation.locked_content_sha256)")
print(f"oot_evaluation sha256 : {OOTEV_SHA_BEFORE}")
print(f"E13 rule selection    : {EV0['selected_model']} ({EV0['status']}) - {EV0['selection_reason']}")

In [0]:
# Only reads what E13 recorded; nothing is recomputed
def rule_status(r):
    if "status" in r:
        return str(r["status"]).upper()
    return "PASS" if r.get("pass") else "FAIL"

sel, over = OV["selected_model"], OV["overrides_rule_selection_of"]
assert sel in MODELS, f"STOP: selected_model {sel!r} is not one of {MODELS}"
assert over == EV0["selected_model"], (
    f"STOP: overrides_rule_selection_of = {over} but the registry's E13 selection is {EV0['selected_model']}")
assert sel != over, "STOP: selected_model equals the rule selection; no override is needed"
assert sel in EV0.get("rules", {}), f"STOP: no recorded rule results for model {sel} in oot_evaluation.rules"

RULE_RESULTS = {r: rule_status(EV0["rules"][sel][r]) for r in ("R1", "R2", "R3", "R4")}
rec_failed  = sorted(r for r, s in RULE_RESULTS.items() if s == "FAIL")
rec_pending = sorted(r for r, s in RULE_RESULTS.items() if s not in ("PASS", "FAIL"))
assert sorted(OV["rules_failed"]) == rec_failed, (
    f"STOP: rules_failed {OV['rules_failed']} differs from the rules E13 recorded as FAIL for {sel}: {rec_failed}")

print(f"Recorded rule results for {sel}: {RULE_RESULTS}")
print(f"rules_failed matches the record: {rec_failed}")
if rec_pending:
    print(f"NOTE: {sel} has rule(s) still PENDING in E13: {rec_pending} - recorded in the section, not resolved here")

In [0]:
# Each ref is "<path>" or "<path>: <sheet>". The file must exist; if the ref does not end in .xlsx and is not
# found, "<path>.xlsx" is accepted and recorded as resolved_path. A named sheet must exist in the workbook.
EVIDENCE = []
for ref in OV["evidence_refs"]:
    path, _, sheet = str(ref).partition(": ")
    path, sheet = path.strip(), sheet.strip() or None
    resolved = path if os.path.exists(path) else (path + ".xlsx" if not path.lower().endswith(".xlsx")
                                                   and os.path.exists(path + ".xlsx") else None)
    item = {"ref": ref, "path": path, "resolved_path": resolved, "sheet": sheet, "exists": resolved is not None,
            "sheet_exists": None, "sha256": None}
    if resolved:
        item["sha256"] = sha256_file(resolved)
        if sheet:
            wb = openpyxl.load_workbook(resolved, read_only=True)
            item["sheet_exists"] = sheet in wb.sheetnames
            wb.close()
    EVIDENCE.append(item)
    print(f"  exists={item['exists']} sheet_ok={item['sheet_exists']} | {resolved or path}"
          + (f" [{sheet}]" if sheet else "") + ("  (resolved: .xlsx added)" if resolved and resolved != path else ""))

bad_ev = [e["ref"] for e in EVIDENCE if not e["exists"] or e["sheet_exists"] is False]
assert not bad_ev, f"STOP: evidence reference(s) not found: {bad_ev}"

In [0]:
SECTION_NEW = {
    **OV,                                                    # the 8 approved fields, as in override.yaml
    "recorded_at": RUN_TIME,
    "git_commit": GIT_COMMIT,
    "config_sha256": CONFIG_HASH,
    "override_file": {"path": "config/override.yaml", "sha256": OVERRIDE_FILE_SHA},
    "registry_version": REG0["version"],
    "locked_content_sha256": LOCKED_SHA_BEFORE,
    "rule_selection": {"selected_model": EV0["selected_model"], "status": EV0["status"],
                       "selection_reason": EV0["selection_reason"], "oot_evaluation_run_time": EV0["run_time"]},
    "recorded_rule_results_of_selected": RULE_RESULTS,
    "rules_pending": rec_pending,
    "evidence": EVIDENCE,
    "module": "s05b_model_choice_override",
}

EXISTING = REG0.get(SECTION)
if EXISTING is None:
    ACTION = "write"
else:
    diffs = [f for f in FIELDS if json.dumps(EXISTING.get(f), sort_keys=True, default=str)
             != json.dumps(SECTION_NEW[f], sort_keys=True, default=str)]
    if diffs:
        raise RuntimeError(
            f"REFUSED: {SECTION} already exists (recorded {EXISTING.get('recorded_at')}, approved by "
            f"{EXISTING.get('approved_by')} on {EXISTING.get('approved_date')}) and field(s) {diffs} differ from "
            f"config/override.yaml. A change needs a new entry with its own approval, kept as history.")
    ACTION = "unchanged"
print(f"Action: {ACTION}")

In [0]:
BACKUP_PATH = None
if ACTION == "write":
    os.makedirs(HISTORY_DIR, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    BACKUP_PATH = os.path.join(HISTORY_DIR, f"model_registry_v{REG0['version']}_before_override_{stamp}.json")
    shutil.copyfile(REGISTRY_PATH, BACKUP_PATH)
    assert sha256_file(BACKUP_PATH) == FILE_SHA_BEFORE, "Backup copy differs from the registry"
    REG1 = {**REG0, SECTION: SECTION_NEW}                    # same keys and values, one key appended
    write_json(REG1, REGISTRY_PATH)
    print(f"Backup : {BACKUP_PATH}")
    print(f"Written: {REGISTRY_PATH} (+ {SECTION})")
else:
    print(f"{SECTION} already present with identical approved fields; registry not rewritten.")

with open(REGISTRY_PATH) as fh:
    REG2 = json.load(fh)
LOCKED_SHA_AFTER = common.registry_locked_sha256(REG2)
OOTEV_SHA_AFTER  = sha256_obj(REG2["oot_evaluation"])
print(f"Locked content sha256 before/after: {LOCKED_SHA_BEFORE[:16]} / {LOCKED_SHA_AFTER[:16]}")
print(f"oot_evaluation sha256 before/after: {OOTEV_SHA_BEFORE[:16]} / {OOTEV_SHA_AFTER[:16]}")
SEL_USED = common.resolve_selected_model(REG2)

In [0]:
import mlflow
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
with mlflow.start_run(run_name="s05b_model_choice_override") as run:
    mlflow.set_tags({"module": "s05b_model_choice_override", "git_commit": GIT_COMMIT,
                     "registry_version": REG2["version"], "action": ACTION})
    mlflow.log_params({"config_sha256": CONFIG_HASH, "selected_model": SEL_USED["model"],
                       "selected_model_source": SEL_USED["source"], "rule_selected_model": SEL_USED["rule_selected_model"],
                       "approved_by": OV["approved_by"], "approved_date": OV["approved_date"],
                       "rules_failed": ",".join(OV["rules_failed"]), "rules_pending": ",".join(rec_pending) or "none",
                       "locked_content_sha256": LOCKED_SHA_AFTER})
    mlflow.log_artifact(OVERRIDE_PATH, artifact_path="config")
    mlflow.log_dict(REG2[SECTION], "model_choice_override.json")
    MLFLOW_RUN_ID = run.info.run_id
print(f"MLflow run: {MLFLOW_RUN_ID}")

In [0]:
with open(REGISTRY_PATH) as fh:
    REG3 = json.load(fh)
sec = REG3.get(SECTION) or {}
extra_keys = sorted(set(REG3) - set(REG0) - {SECTION})
missing_keys = sorted(set(REG0) - set(REG3))

CHECKS = [
    ("Locked content unchanged (sha256 before = after = oot_evaluation.locked_content_sha256)",
     common.registry_locked_sha256(REG3) == LOCKED_SHA_BEFORE == REG3["oot_evaluation"]["locked_content_sha256"]),
    ("oot_evaluation unchanged (sha256 before = after)", sha256_obj(REG3["oot_evaluation"]) == OOTEV_SHA_BEFORE),
    ("Registry still locked: true with the same locked_at", REG3.get("locked") is True and REG3.get("locked_at") == REG0["locked_at"]),
    (f"No other key added or removed (added: {extra_keys}, removed: {missing_keys})", not extra_keys and not missing_keys),
    (f"{SECTION} present with all {len(FIELDS)} fields non-empty", bool(sec) and all(not _empty(sec.get(f)) for f in FIELDS)),
    (f"{SECTION} fields equal config/override.yaml", all(json.dumps(sec.get(f), default=str) == json.dumps(OV[f], default=str) for f in FIELDS)),
    ("Timestamp, git commit and config hash recorded", all(str(sec.get(k, "")).strip() for k in ("recorded_at", "git_commit", "config_sha256"))),
    ("Every evidence reference exists (and named sheets exist)", all(e["exists"] and e["sheet_exists"] is not False for e in sec.get("evidence", []))),
    (f"resolve_selected_model uses the override ({SEL_USED['model']} from {SEL_USED['source']})",
     SEL_USED["model"] == OV["selected_model"] and SEL_USED["source"] == f"{SECTION}.selected_model"),
    ("Locked model files re-verified after the write", all(sha256_file(v["path"]) == v["sha256"] for v in REG3["files"].values())),
]
for name, ok in CHECKS:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
ALL_PASS = all(ok for _, ok in CHECKS)

os.makedirs(RUN_SUMS, exist_ok=True)
write_json({
    "module": "s05b_model_choice_override", "run_time": RUN_TIME, "duration_s": round(time.time() - _t0, 1),
    "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH, "branch": RUN_CTX["branch"],
    "registry": REGISTRY_PATH, "registry_version": REG3["version"], "action": ACTION, "backup": BACKUP_PATH,
    "row_counts": {"evidence_refs": len(EVIDENCE), "registry_keys": len(REG3)},
    "selected_model": SEL_USED["model"], "selected_model_source": SEL_USED["source"],
    "rule_selected_model": SEL_USED["rule_selected_model"], "rules_failed": OV["rules_failed"], "rules_pending": rec_pending,
    "locked_content_sha256": {"before": LOCKED_SHA_BEFORE, "after": common.registry_locked_sha256(REG3)},
    "oot_evaluation_sha256": {"before": OOTEV_SHA_BEFORE, "after": sha256_obj(REG3["oot_evaluation"])},
    "registry_file_sha256": {"before": FILE_SHA_BEFORE, "after": sha256_file(REGISTRY_PATH)},
    "mlflow_run_id": MLFLOW_RUN_ID,
    "checks": [{"check": n, "pass": bool(ok)} for n, ok in CHECKS], "pass": ALL_PASS,
}, SUMMARY_PATH)
print(f"Run summary: {SUMMARY_PATH}")

for name, ok in CHECKS:
    assert ok, f"Acceptance FAILED: {name}"
print(f"All {len(CHECKS)} acceptance checks PASS")